### RaschPy model selection worked example

RaschPy provides three model-selection tools for choosing the simplest model structure that adequately fits the data:

1. **RSM vs PCM** (`model_selection()`, on either model) — is a single shared threshold structure (RSM) adequate, or does the data need independent per-item thresholds (PCM)?
2. **MFRM rater parameterisation** (`model_selection()`, on `MFRM`) — of the five rater severity structures (global, items, thresholds, bivector, matrix), which fits best?
3. **MFRM "mixed" model** (`per_rater_model_selection()`) — rather than forcing every rater into the same parameterisation, assign each rater individually the simplest structure that fits their own behaviour.

All three support likelihood-ratio, AIC, and BIC criteria. This notebook simulates data under a known structure in each case and checks whether the simpler tools recover it.

#### RSM vs PCM

In [1]:
import raschpy as rp

Simulate data genuinely generated under RSM (one threshold structure shared by all items) and check whether `model_selection()` (available on either `RSM` or `PCM`) correctly prefers RSM:

In [2]:
sim_rsm = rp.RSM_Sim(no_of_items=15, no_of_persons=1500, max_score=4, seed=1)
rsm = rp.RSM(sim_rsm)
rsm.calibrate()
rsm.model_selection(test='AIC')
rsm.model_comparison_rsm_pcm_aic_summary

PCM AIC      26636.199713
RSM AIC      26606.693343
p-value               0.0
Preferred             RSM
Name: RSM vs PCM AIC comparison, dtype: object

Now simulate data genuinely generated under PCM (each item drawing its own, independent threshold structure) and check that `model_selection()` correctly prefers PCM instead:

In [3]:
sim_pcm = rp.PCM_Sim(no_of_items=15, no_of_persons=2000, max_score_vector=[4] * 15, category_base=3, seed=1)
pcm = rp.PCM(sim_pcm)
pcm.calibrate()
pcm.model_selection(test='AIC')
pcm.model_comparison_rsm_pcm_aic_summary

PCM AIC      20402.942383
RSM AIC       24469.01155
p-value               0.0
Preferred             PCM
Name: RSM vs PCM AIC comparison, dtype: object

`test='LR'` and `test='BIC'` are also available — results are stored as `model_comparison_rsm_pcm_{lr,aic,bic}_summary` respectively (plus `_preferred`, and for AIC with `aic_sig_test=True`, a relative-likelihood `_aic_p`).

#### MFRM rater-parameterisation model selection

Simulate data genuinely generated under the "items" rater parameterisation (a separate severity per rater x item combination) and check that `model_selection()` (which calibrates all five parameterisations and ranks them) correctly identifies it:

In [4]:
sim_mfrm = rp.MFRM_Sim_Items(no_of_items=10, no_of_persons=500, no_of_raters=8, max_score=4,
                              item_range=3, facet_range=2, seed=1)
mfrm = rp.MFRM(sim_mfrm)
mfrm.calibrate_global()   # any one calibration is enough to instantiate the model
mfrm.model_selection(test='AIC')
mfrm.model_comparison_mfrm_aic_summary

,LL,k,AIC,ΔAIC
Model,,,,
items,-43719.706720,82.0,87603.413440,-3803.926111
bivector,-43730.699684,103.0,87667.399367,-3739.940184
matrix,-43635.061317,292.0,87854.122634,-3553.216917
thresholds,-45651.837805,40.0,91383.675611,-23.663941
global,-45684.669776,19.0,91407.339551,0.000000


In [5]:
mfrm.model_comparison_mfrm_aic_preferred

'items'

#### MFRM mixed rater model

Real rating designs rarely have every rater behaving with the same complexity — some might be simple, uniformly-shifted raters (`'global'`), while others show genuine item-by-item variation (`'items'`). Rather than forcing one parameterisation on everyone, `per_rater_model_selection()` assigns each rater individually, via a top-down testing ladder (matrix → bivector → items/thresholds → global).

Simulate 8 raters: 4 behave as simple `'global'` raters (a single severity shift, no further structure), and 4 show a genuine per-item "halo" pattern (systematically more lenient on some items, more severe on others) — i.e. genuinely `'items'`-parameterised behaviour.

In [6]:
max_score = 4
no_of_items = 8

flat = {f'Item_{i + 1}': [0] * max_score for i in range(no_of_items)}
halo = {f'Item_{i + 1}': [((no_of_items - 1) / 2) - i] * max_score for i in range(no_of_items)}

sim_mixed = rp.MFRM_Sim_Matrix(
    no_of_items=no_of_items, no_of_persons=2000, no_of_raters=8, max_score=max_score, person_sd=4,
    manual_raters={'Rater_1': flat, 'Rater_2': flat, 'Rater_3': flat, 'Rater_4': flat,
                   'Rater_5': halo, 'Rater_6': halo, 'Rater_7': halo, 'Rater_8': halo},
    seed=1,
)
generating_models = {'Rater_1': 'global', 'Rater_2': 'global', 'Rater_3': 'global', 'Rater_4': 'global',
                     'Rater_5': 'items', 'Rater_6': 'items', 'Rater_7': 'items', 'Rater_8': 'items'}

mfrm_mixed = rp.MFRM(sim_mixed)
mfrm_mixed.calibrate_matrix()   # calibrates the full matrix model once; everything else is derived from it, no refitting
mfrm_mixed.per_rater_model_selection(min_effect=0.3)

,selected_model,LL_matrix,LL_bivector,LL_items,LL_thresholds,LL_global,AIC_matrix,AIC_bivector,AIC_items,AIC_thresholds,AIC_global,p_vs_global
Rater,,,,,,,,,,,,
Rater_1,global,-3583.641071,-3586.470591,-3590.462364,-3412.938378,-3416.782475,7231.282142,7194.941181,7196.924729,6833.876756,6835.564951,0.429945
Rater_2,global,-3674.763133,-3677.452118,-3673.277327,-3486.555463,-3482.699678,7413.526267,7376.904236,7362.554654,6981.110926,6967.399355,1.0
Rater_3,global,-3551.610556,-3558.213867,-3564.429779,-3390.950171,-3396.880361,7167.221113,7138.427733,7144.859558,6789.900342,6795.760723,0.053387
Rater_4,global,-3850.010438,-3847.789074,-3846.160835,-3583.543178,-3582.005539,7764.020876,7717.578147,7708.32167,7175.086355,7166.011078,1.0
Rater_5,items,-2199.72448,-2202.232414,-2202.151498,-3587.357866,-3587.569938,4463.448959,4426.464829,4420.302997,7182.715732,7177.139877,0.0
Rater_6,items,-2159.210754,-2160.565425,-2153.772686,-3593.224074,-3588.845112,4382.421509,4343.13085,4323.545372,7194.448147,7179.690224,0.0
Rater_7,items,-2222.320757,-2221.911652,-2223.22977,-3551.142194,-3551.847475,4508.641515,4465.823305,4462.45954,7110.284387,7105.69495,0.0
Rater_8,items,-2270.300394,-2271.746134,-2276.423822,-3651.740673,-3654.608481,4604.600789,4565.492268,4568.847643,7311.481345,7311.216962,0.0


`rater_models` gives the assigned parameterisation per rater; `per_rater_model_selection_counts` summarises how many raters landed on each:

In [7]:
mfrm_mixed.rater_models

Rater
Rater_1    global
Rater_2    global
Rater_3    global
Rater_4    global
Rater_5     items
Rater_6     items
Rater_7     items
Rater_8     items
Name: selected_model, dtype: object

In [8]:
mfrm_mixed.per_rater_model_selection_counts

selected_model
global    4
items     4
Name: Count, dtype: int64

Compare against the true generating parameterisation for each rater:

In [9]:
import pandas as pd

comparison = pd.DataFrame({
    'Generating': pd.Series(generating_models),
    'Selected': mfrm_mixed.rater_models,
})
comparison['Correct'] = comparison['Generating'] == comparison['Selected']
comparison

,Generating,Selected,Correct
Rater_1,global,global,True
Rater_2,global,global,True
Rater_3,global,global,True
Rater_4,global,global,True
Rater_5,items,items,True
Rater_6,items,items,True
Rater_7,items,items,True
Rater_8,items,items,True


With every rater correctly assigned, fit statistics and rater statistics for the mixed model can be generated exactly as for any single parameterisation, just passing `model='mixed'`:

In [10]:
mfrm_mixed.fit_statistics(model='mixed')
mfrm_mixed.rater_stats_df(model='mixed', full=True)
mfrm_mixed.rater_stats_mixed.head()

Model   Item_1          Item_2          Item_3          Item_4  \
                Estimate     SE Estimate     SE Estimate     SE Estimate   
Rater_1  global   -0.029  0.026   -0.029  0.026   -0.029  0.026   -0.029   
Rater_2  global   -0.024  0.025   -0.024  0.025   -0.024  0.025   -0.024   
Rater_3  global   -0.035  0.026   -0.035  0.026   -0.035  0.026   -0.035   
Rater_4  global    0.027  0.025    0.027  0.025    0.027  0.025    0.027   
Rater_5   items    1.990  0.093    1.307  0.068    0.623  0.058    0.260   

                 Item_5  ... Threshold 2 Threshold 3      Threshold 4       \
            SE Estimate  ...          SE    Estimate   SE    Estimate   SE   
Rater_1  0.026   -0.029  ...         0.0         0.0  0.0         0.0  0.0   
Rater_2  0.025   -0.024  ...         0.0         0.0  0.0         0.0  0.0   
Rater_3  0.026   -0.035  ...         0.0         0.0  0.0         0.0  0.0   
Rater_4  0.025    0.027  ...         0.0         0.0  0.0         0.0  0.0   
Rater_5  0.048   -0.242  ...         0.0         0.0  0.0         0.0  0.0   

        Overall statistics                                      
                     Count Infit MS Infit Z Outfit MS Outfit Z  
Rater_1            16000.0    1.037   2.145     0.928   -0.999  
Rater_2            16000.0    1.022   1.285     0.909   -1.274  
Rater_3            16000.0    1.044   2.546     0.963   -0.500  
Rater_4            16000.0    1.057   3.247     0.994   -0.062  
Rater_5            16000.0    0.714 -17.589     0.569   -5.584  

[5 rows x 30 columns]

`per_rater_model_selection(anchors=[...])` additionally supports anchoring the matrix calibration to a chosen rater subset before testing — different anchor sets can give different (all individually valid) per-rater assignments, since the zero-sum constraint means a non-uniform rater can induce apparent non-uniformity in others. Anchored results are stored as `anchor_rater_models` / `anchor_facet_effects_mixed` / `anchor_per_rater_model_selection_table` / `anchor_per_rater_model_selection_counts`.